# 第 12 章　変数間の関係

風速と貸出件数の関係を，日付と時計時刻に対応する点から読みます。

このノートの前半は，本文の説明と一緒に実行します。各場面で先に短い問いを読み，講師の案内に合わせてセルを動かし，出力の見所を確認します。後半は約20分の実習です。前半のコードは動く見本として用意してあり，作図コードをすべて書き写す必要はありません。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV は，手元にファイルがなければ公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

file_name = "bike-sharing-hourly.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

hourly = pd.read_csv(csv_path)
hourly[["dteday", "hr", "windspeed", "cnt"]].head(3)

## 本文と一緒に実行

### 1. 同じ風速でも件数は同じか

全時刻の散布図を描きます。風速指標0.1940の二つの点について，日付と時計時刻を表へ戻って確認してください。

In [ ]:
columns = ["dteday", "hr", "windspeed", "cnt"]
same_wind = hourly.loc[hourly["windspeed"] == 0.1940]
low_rentals = same_wind.loc[same_wind["cnt"].idxmin()]
high_rentals = same_wind.loc[same_wind["cnt"].idxmax()]

In [ ]:
plt.rcParams["figure.figsize"] = (6.1, 3.7)
fig, ax = plt.subplots(layout="constrained")
ax.scatter(hourly["windspeed"], hourly["cnt"], s=8, color="#286b70",
           alpha=0.15, linewidths=0)
ax.scatter(low_rentals["windspeed"], low_rentals["cnt"], s=58, color="#a34f30",
           marker="o", edgecolors="white", linewidths=1.2, zorder=3, clip_on=False)
ax.scatter(high_rentals["windspeed"], high_rentals["cnt"], s=58, color="#a34f30",
           marker="s", edgecolors="white", linewidths=1.2, zorder=3)
ax.set(xlabel="Wind-speed index", ylabel="Rentals per hour",
       xlim=(-0.025, 0.875), ylim=(0, 1000))
plt.show()

In [ ]:
print(len(same_wind))
same_wind.loc[[low_rentals.name, high_rentals.name], columns]

In [ ]:
overall_r = hourly["windspeed"].corr(hourly["cnt"])
print(f"{overall_r:+.3f}")

**出力の見所：**同じ風速指標には1,657行あり，貸出件数は1〜943件です。相関+0.093という一つの数と，図に残る縦の広がりの両方を読んでください。

### 2. 四点の組合せを替える

教材用の四点です。二列の値を一つずつ見ると同じ値がそろいます。どのxとどのyを組にするかだけを替えたら，相関は同じになるでしょうか。

In [ ]:
small = pd.DataFrame({"x": [1, 2, 4, 5], "y": [1, 4, 2, 5], "repaired_y": [5, 4, 2, 1]})
small

In [ ]:
plt.rcParams["figure.figsize"] = (6.3, 3.2)
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, layout="constrained")
for ax, column, title in zip(axes, ["y", "repaired_y"], ["Original pairs", "Different pairs"]):
    ax.axhline(3, color="#6f7b86", linestyle="--", linewidth=1)
    ax.axvline(3, color="#6f7b86", linestyle="--", linewidth=1)
    ax.scatter(small["x"], small[column], color="#286b70", s=35, zorder=3)
    for name, x, y in zip("ABCD", small["x"], small[column]):
        ax.annotate(name, (x, y), xytext=(5, 4), textcoords="offset points")
    ax.set(title=title, xlabel="x", xlim=(0.4, 5.7), ylim=(0.4, 5.7),
           xticks=[1, 2, 3, 4, 5], yticks=[1, 2, 3, 4, 5])
axes[0].set_ylabel("y")
plt.show()

In [ ]:
print(round(small["x"].corr(small["y"]), 2))
print(round(small["x"].corr(small["repaired_y"]), 2))

**出力の見所：**相関は+0.60と−1.00です。平均3の線から見て，同じ方向のずれと逆方向のずれが，どの点にあるか説明します。

### 3. 時計時刻をそろえる

8時台と13時台を先に選んで比べます。全時刻の図と軸の範囲を共通にします。条件を加えたとき，どの日付・時刻の点が外れたかも考えてください。

In [ ]:
at_eight = hourly.loc[hourly["hr"] == 8]
at_thirteen = hourly.loc[hourly["hr"] == 13]

r_eight = at_eight["windspeed"].corr(at_eight["cnt"])
r_thirteen = at_thirteen["windspeed"].corr(at_thirteen["cnt"])
print(f"08:00: {len(at_eight)} rows, r={r_eight:+.3f}")
print(f"13:00: {len(at_thirteen)} rows, r={r_thirteen:+.3f}")

In [ ]:
plt.rcParams["figure.figsize"] = (7, 3.4)
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, layout="constrained")
for ax, selected, title in zip(axes, [at_eight, at_thirteen], ["08:00", "13:00"]):
    ax.scatter(selected["windspeed"], selected["cnt"],
               color="#286b70", alpha=0.36, s=13, linewidths=0)
    ax.set(title=title, xlabel="Wind-speed index", xlim=(-0.025, 0.875),
           ylim=(0, 1000), yticks=[0, 250, 500, 750, 1000])
axes[0].set_ylabel("Rentals per hour")
plt.show()

**出力の見所：**8時台は727行で約−0.092，13時台は729行で約−0.086です。符号は変わりましたが，どちらも弱い直線的な関係です。同じ風速指標でも大きな件数の幅が残ります。

### 4. 群の位置を平均点で読む

一つの時刻に属する日々の点を，一つの平均点にまとめます。次の図で「05」「17」は一日の記録でしょうか，多くの日の集計でしょうか。

In [ ]:
hour_means = hourly.groupby("hr")[["windspeed", "cnt"]].mean()
hour_means.loc[[5, 8, 13, 16, 17, 23]].round(3)

In [ ]:
plt.rcParams["figure.figsize"] = (6, 3.8)
fig, ax = plt.subplots(layout="constrained")
ax.scatter(hour_means["windspeed"], hour_means["cnt"], s=28, color="#667f8d")
for hour, offset in {
    5: (-18, 5), 8: (-7, 8), 13: (-4, 11), 16: (5, 6), 17: (6, -2), 23: (6, -2),
}.items():
    mean_wind = hour_means.loc[hour, "windspeed"]
    mean_rentals = hour_means.loc[hour, "cnt"]
    ax.scatter(mean_wind, mean_rentals, marker="s", s=32, color="#a34f30", zorder=3)
    ax.annotate(f"{hour:02d}", (mean_wind, mean_rentals), xytext=offset,
                textcoords="offset points")
ax.set(
    xlabel="Mean wind-speed index", ylabel="Mean rentals per hour",
    xlim=(0.137, 0.252), ylim=(-20, 500),
)
plt.show()

In [ ]:
hour_mean_r = hour_means["windspeed"].corr(hour_means["cnt"])
print(f"Clock-hour averages: {len(hour_means)} points, r={hour_mean_r:+.3f}")

**出力の見所：**24個の時刻別平均の相関は約+0.817です。日々の点の広がりは平均にまとめられています。この係数を，同じ時刻の日々の予測の強さと読み替えられるか考えます。

### 5. 時刻の中に残る条件

同じ8時台でも，土日・祝日とそれ以外では何が違いそうでしょうか。色と形で暦の区分を表示します。

In [ ]:
calendar_means = (
    hourly.loc[hourly["hr"].isin([8, 13])]
    .groupby(["hr", "workingday"])["cnt"]
    .agg(rows="size", mean="mean")
)
calendar_means.round(1)

In [ ]:
plt.rcParams["figure.figsize"] = (7, 3.8)
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, layout="constrained")
for ax, selected, title in zip(axes, [at_eight, at_thirteen], ["08:00", "13:00"]):
    for flag, label, marker, color in [
        (0, "Weekend / holiday", "s", "#a34f30"),
        (1, "Other weekday", "o", "#286b70"),
    ]:
        group = selected.loc[selected["workingday"] == flag]
        ax.scatter(group["windspeed"], group["cnt"], label=label,
                   color=color, marker=marker, alpha=0.38, s=13, linewidths=0)
    ax.set(title=title, xlabel="Wind-speed index", xlim=(-0.025, 0.875),
           ylim=(0, 1000), yticks=[0, 250, 500, 750, 1000])
axes[0].set_ylabel("Rentals per hour")
axes[1].legend(frameon=False, loc="upper right", fontsize=8, markerscale=1.4)
plt.show()

**出力の見所：**8時台は土日・祝日以外が高く，13時台は土日・祝日が高い傾向です。風だけで貸出件数が決まるわけではなく，時刻をそろえた後にも他の条件が残ります。

## Jupyter 実習（約20分）

ここからは，夕方の17時台を自分で選びます。講義の8時台・13時台の変数は課題の解答として使わず，`chosen` と `chosen_r` を以下で作ります。

## 1. 全時刻の図から予想する（約 5 分）

動く例として，収録された全時刻を一枚の散布図にします。横軸は風速指標，縦軸は貸出件数です。全体の相関も表示します。

貸出に備える時刻として 17 時台を選ぶ理由を一文で書きます。17 時台だけを取り出すと，点の縦の広がりや斜めの向きは，どのように変わると予想しますか。結果を調べる前に，一箇所見たい場所を書きます。例えば風速指標 0.1940 付近の縦の幅を比べられます。

（ここに理由と予想を記入）

In [ ]:
overall_r = hourly["windspeed"].corr(hourly["cnt"])
print(f"All hours: {len(hourly):,} rows, r={overall_r:+.3f}")

x_limits = (-0.025, 0.875)
y_limits = (0, 1000)
fig, ax = plt.subplots(layout="constrained")
ax.scatter(hourly["windspeed"], hourly["cnt"], color="#286b70",
           alpha=0.15, s=8, linewidths=0)
ax.set(xlabel="Wind-speed index", ylabel="Rentals per hour",
       xlim=x_limits, ylim=y_limits)
plt.show()

## 2. 時計時刻をそろえる（約 9 分）

最初のセルで，`hourly` から `hr` が `target_hour` の行を選んで `chosen` に入れます。`target_hour` は 17 です。表示された件数と最初の 3 行から，日付は違っても時計時刻がそろっていることを確かめてください。

次のセルで，`chosen` の風速指標と貸出件数の相関を `chosen_r` に入れます。条件を作る一行と相関を求める一行を記入すると，二枚の図が同じ軸で表示されます。

In [ ]:
target_hour = 17
chosen = None
# target_hour に合う行を選び，chosen に入れる。

if chosen is None:
    print("時刻を指定する一行を書いてから，このセルを実行します。")
else:
    print(f"{len(chosen):,} rows at {target_hour:02d}:00")
    display(chosen[["dteday", "hr", "windspeed", "cnt"]].head(3))

In [ ]:
chosen_r = None
if chosen is not None:
    # chosen の同じ行の二列の相関を chosen_r に入れる。
    pass

if chosen_r is None:
    print("時刻の選択と相関の両方を書くと，比較図が出ます。")
else:
    print(f"All hours: r={overall_r:+.3f}")
    print(f"{target_hour:02d}:00: r={chosen_r:+.3f}")
    fig, axes = plt.subplots(
        1, 2, figsize=(8, 3.4), sharex=True, sharey=True, layout="constrained"
    )
    for ax, data, title, alpha, size in [
        (axes[0], hourly, "All hours", 0.15, 8),
        (axes[1], chosen, f"{target_hour:02d}:00", 0.36, 13),
    ]:
        ax.scatter(data["windspeed"], data["cnt"], color="#286b70",
                   alpha=alpha, s=size, linewidths=0)
        ax.set(title=title, xlabel="Wind-speed index", xlim=x_limits, ylim=y_limits)
    axes[0].set_ylabel("Rentals per hour")
    plt.show()

<details>
<summary>二つの行の書き方を確認する</summary>

`chosen = None` の下に，対象時刻に当てはまる行を選ぶ行を書きます。

```python
chosen = hourly.loc[hourly["hr"] == target_hour]
```

`if chosen is not None:` の中で，次の一行を書きます。

```python
    chosen_r = chosen["windspeed"].corr(chosen["cnt"])
```

</details>

## 3. 夕方の貸出について報告する（約 6 分）

最初の予想と見比べ，夕方の準備をする担当者に 3〜4 文で伝えます。

1. 全体と 17 時台の行数・相関はどう違ったか。それぞれ，どの日付と時刻の組を比べたか。
2. 17 時台の点の集まりで，同じくらいの風速でも件数がどの程度違うか。図で読んだ位置か範囲を一つ挙げる。
3. 明日の貸出件数を考えるために追加で知りたい条件を一つ選び，なぜそれを選ぶか説明する。

（ここに報告を記入）

カーネルを再起動し，上から全セルを実行します。図と説明を残したノートブックを提出します。

### 土日・祝日以外の 17 時台（任意）

同じ `chosen` から `workingday` が `1` の行に絞ります。同じ時計時刻で暦の区分を揃えると，どのような日付が比較に残りますか。行数と相関を調べ，必須課題で作った 17 時台の説明にどんな情報を添えられるか考えます。

### 出典

Capital Bikeshare の時間別集計と気象・暦の情報は，Hadi Fanaee-T が UCI Machine Learning Repository で公開した `hour.csv` を教材用に改名したものです。 [データ配布元](https://doi.org/10.24432/C5W894) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)